In [0]:

# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Bronze
# Notebook        : bronze_purchase_requisition
# Source          : purchase_requisition.csv
# Target          : procurement.bronze.bronze_purchase_requisition
# Audit Table     : procurement.audit.duplicate_purchase_requisition
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads raw purchase_requisition master data into the Bronze layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Load purchase_requisition master data from the source CSV file into the Bronze layer.
#
# Preserve the raw source data with minimal transformations.
#
# Detect duplicate Payment IDs and store them in the Audit schema for business review.
#
# Add audit columns to support data lineage and traceability.
#
# Create a reliable Bronze Delta table that will serve as the source for the Silver layer.
#
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
print(CATALOG)
print(BRONZE_PURCHASE_REQUISITIONS)
print(AUDIT_DUPLICATE_PURCHASE_REQUISITIONS)
print(PURCHASE_REQUISITIONS_FILE)

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
#Purchase_Requisition schema
purchase_requisition_schema = StructType([
    StructField("pr_id", StringType(), False),
    StructField("pr_date", StringType(), True),
    StructField("employee_id", StringType(), True),
    StructField("department_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity_requested", IntegerType(), True),
    StructField("estimated_unit_price", DecimalType(18,2), True),
    StructField("estimated_total", DecimalType(18,2), True),
    StructField("priority",StringType(),True),
    StructField("status", StringType(), True) ,
    StructField("approver_id", StringType(), True) ,
    StructField("approval_date", StringType(), True)
  ])
# Read Purchase_Requisition master data from landing volume
bronze_purchase_requisition_df = (spark.read
    .format("csv")
    .option("header", True)
    .schema(purchase_requisition_schema)
    .load(PURCHASE_REQUISITIONS_FILE)
)

#Source Data validation 
print(f"Total Records : {bronze_purchase_requisition_df.count()}")

print("\nSchema:")
bronze_purchase_requisition_df.printSchema()

print("\nColumns:")
print(bronze_purchase_requisition_df.columns)

print("\nSampledata:")
display(bronze_purchase_requisition_df.limit(10))

In [0]:
# Check the NULL and Blank pr_ids
null_blank_pr_id = bronze_purchase_requisition_df.filter(col("pr_id").isNull() | (trim(col("pr_id")) == ""))

print(f"Total NULL or Blank pr_ids : {null_blank_pr_id.count()}")

display(null_blank_pr_id)

In [0]:
#Check the NULL and Blank pr date
null_blank_pr_date = bronze_purchase_requisition_df.filter(col("pr_date").isNull() | (trim(col("pr_date")) == ""))

print(f"Total NULL or Blank pr_date : {null_blank_pr_date.count()}")

display(null_blank_pr_date)

In [0]:
# Check the NULL and Blank employee_ids
null_blank_employee_id = bronze_purchase_requisition_df.filter(col("employee_id").isNull() | (trim(col("employee_id")) == ""))

print(f"Total NULL or Blank employee_ids : {null_blank_employee_id.count()}")

display(null_blank_employee_id)

In [0]:
# Check the NULL and Blank department_ids
null_blank_department_id = bronze_purchase_requisition_df.filter(col("department_id").isNull() | (trim(col("department_id")) == ""))

print(f"Total NULL or Blank department_id : {null_blank_department_id.count()}")

display(null_blank_department_id)

In [0]:
# Check the NULL and Blank product_ids
null_blank_product_id = bronze_purchase_requisition_df.filter(col("product_id").isNull() | (trim(col("product_id")) == ""))

print(f"Total NULL or Blank product_ids : {null_blank_product_id.count()}")

display(null_blank_product_id)

In [0]:
#Check the NUll and negative estimated_unit_price
negative_estimated_unit_price = bronze_purchase_requisition_df.filter((col("estimated_unit_price") < 0) | (col("estimated_unit_price").isNull()))

print(f"Total Null and negative estimated_unit_price : {negative_estimated_unit_price.count()}")

display(negative_estimated_unit_price)

In [0]:
#Check the NUll and negative estimated_total
negative_estimated_total = bronze_purchase_requisition_df.filter((col("estimated_total") < 0) | (col("estimated_total").isNull()))

print(f"Total Null and negative estimated_total : {negative_estimated_total.count()}")

display(negative_estimated_total)

In [0]:
#Check the NULL and Blank priority
null_blank_priority = bronze_purchase_requisition_df.filter(col("priority").isNull() | (trim(col("priority")) == ""))

print(f"Total NULL or Blank priority : {null_blank_priority.count()}")

display(null_blank_priority)

In [0]:
#Check the NULL and Blank status
null_blank_status = bronze_purchase_requisition_df.filter(col("status").isNull() | (trim(col("status")) == ""))

print(f"Total NULL or Blank status : {null_blank_status.count()}")

display(null_blank_status)

In [0]:
#Check the NULL and Blank approver_id
null_blank_approver_id = bronze_purchase_requisition_df.filter(col("approver_id").isNull() | (trim(col("approver_id")) == ""))

print(f"Total NULL or Blank approver_id : {null_blank_approver_id.count()}")

display(null_blank_approver_id)

In [0]:
#Check the NULL and Blank approver_id
null_blank_approver_id = bronze_purchase_requisition_df.filter(col("approver_id").isNull() | (trim(col("approver_id")) == ""))

print(f"Total NULL or Blank approver_id : {null_blank_approver_id.count()}")

display(null_blank_approver_id)

In [0]:
#Check the NULL and Blank approval_date
null_blank_approval_date = bronze_purchase_requisition_df.filter(col("approval_date").isNull() | (trim(col("approval_date")) == ""))

print(f"Total NULL or Blank approval_date : {null_blank_approval_date.count()}")

display(null_blank_approval_date)

In [0]:
# ============================================================
# Identify Duplicate purchase_requisition IDs
# ============================================================

duplicate_purchase_requisition_keys = (
    bronze_purchase_requisition_df
        .groupBy("pr_id")
        .count()
        .filter(col("count") > 1)
        .withColumnRenamed("count", "duplicate_IDs")
)

display(duplicate_purchase_requisition_keys)

In [0]:
# ============================================================
# Identify Duplicate purchase_requisition Records
# Business Rule: Keep the first occurrence of each PR ID and identify subsequent records as duplicates.
# ============================================================

window_spec = Window.partitionBy("pr_id").orderBy("pr_id")

purchase_requisition_rank_df = (
    bronze_purchase_requisition_df
        .join(
            duplicate_purchase_requisition_keys.select("pr_id"),
            on="pr_id",
            how="inner"
        )
        .withColumn(
            "row_num",
            row_number().over(window_spec)
        )
)
display(purchase_requisition_rank_df)

In [0]:
# ============================================================
# Retrieve Duplicate  Payment Records
# ============================================================

duplicate_purchase_requisition = (
    purchase_requisition_rank_df
        .filter(col("row_num") > 1)
        .drop("row_num")
)

print(f"Duplicate Purchase_Requisition Records : {duplicate_purchase_requisition.count()}")
display(duplicate_purchase_requisition)


In [0]:
# ============================================================
# Add Audit Metadata for duplicate PO IDs
# ============================================================

duplicate_purchase_requisition = (
    duplicate_purchase_requisition
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("Purchase_Requisition"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(duplicate_purchase_requisition)

In [0]:
# ============================================================
# Add Audit Metadata for NULL PR IDs
# ============================================================

from pyspark.sql.functions import current_timestamp, lit

invalid_purchase_requisition = (
    null_blank_pr_id
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("purchase_requisition"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(invalid_purchase_requisition)

In [0]:
# ============================================================
# Write Duplicate Records to Audit Table
# ============================================================

duplicate_count = duplicate_purchase_requisition.count()

if duplicate_count > 0:

    write_delta(
        df = duplicate_purchase_requisition,
         table_name = AUDIT_DUPLICATE_PURCHASE_REQUISITIONS
    )

    print(f"Successfully written {duplicate_count} duplicate record(s) to {AUDIT_DUPLICATE_PURCHASE_REQUISITIONS}")

else:

    print("No duplicate Purchase_Requisition records found. Audit table not created.")

In [0]:
# ============================================================
# Write Invalid NULL Records to Audit Table
# ============================================================

invalid_count = invalid_purchase_requisition.count()

if invalid_count > 0:

    write_delta(
        df = invalid_purchase_requisition,
         table_name = AUDIT_INVALID_PURCHASE_REQUISITIONS
    )

    print(f"Successfully written {invalid_count} invalid record(s) to {AUDIT_INVALID_PURCHASE_REQUISITIONS}")

else:

    print("No invalid purchase_requisition records found. Audit table not created.")

In [0]:
# ============================================================
# Add Bronze Audit Columns
# ============================================================

bronze_purchase_requisition_final_df = (
    bronze_purchase_requisition_df
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("source_file", lit("Purchase_Requisition.csv"))
)
display(bronze_purchase_requisition_final_df)

In [0]:
# ============================================================
# Write Bronze Delta Table
# ============================================================

write_delta(df=bronze_purchase_requisition_final_df,table_name=BRONZE_PURCHASE_REQUISITIONS)

In [0]:
# ============================================================
# Validate Bronze Delta Table
# ============================================================

bronze_purchase_requisition = spark.table(BRONZE_PURCHASE_REQUISITIONS)

print(f"Total Bronze Records : {bronze_purchase_requisition.count()}")

display(bronze_purchase_requisition)

In [0]:
# ============================================================
# Bronze purchase_requisition complete summary
# ============================================================
print("=" * 60)
print("Bronze Purchase_Requisition  Load Completed Successfully")
print("=" * 60)

print(f"{'Landing Records':<30}: {bronze_purchase_requisition_df.count()}")

print(f"{'Duplicate Audit Records':<30}: {duplicate_purchase_requisition.count()}")

print(f"{'Invalid Invoice Records':<30}: {invalid_purchase_requisition.count()}")

print(f"{'Total Audit Records':<30}: {duplicate_purchase_requisition.count() + invalid_purchase_requisition.count()}")

print(f"{'Bronze Records':<30}: {spark.table(BRONZE_PURCHASE_REQUISITIONS).count()}")